# Адаптер on-policy: правки + свои пары «решил / не решил» (Colab A100 80 ГБ)

Два вида пар (DPO с reference, `--ref`):
* `pref_edit` (188 пар, как у `pref_edit_v1`, который дал 0.15 -> 0.20): правильный / сломанный `edit_file`;
* `onpolicy` (`train.build_pref_onpolicy`, 62 пары: 38 из публичных логов + 24 из MiMo): задачи, которые модель
  решает через раз. chosen -- её же первая верная правка из решённого прогона, rejected -- неверная правка
  (или repro-обход) из проваленного прогона на том же ходу. Это собственные ответы модели, а не чужие.

На Drive (`MyDrive/gemma_adapters/`): исходная отправка (`submissions/v5.zip`), `pref_edit.jsonl`,
`pref_onpolicy.jsonl` (из `data/pref_onpolicy.jsonl` ноутбука). Задачи `eval/mimo50.json` в пары не попали.

Ячейки: **0** настройки, **1** GPU, **2** код, **3** Drive и пары, **4** модель, **5** обучение (vLLM не запускать),
**6** сохранение, **7** архив отправки.

## 0. Параметры (правьте только здесь)

In [ ]:
SUBMISSION_ZIP = "submissions/v5.zip"        # отправка v5: промпт и eval_config (путь внутри gemma_adapters)
MAX_TIME_MINUTES = 5.5                       # с 8 мин v5 не укладывалась в лимит Kaggle
PAIRS = "pref_edit.jsonl"                    # пары правок (188)
ONPOLICY = "pref_onpolicy.jsonl"             # пары on-policy (62)
ADAPTER = "pref_onpol_v1"
EPOCHS = 1
EDIT_REPEAT = 2                              # пары правок -- 2 раза за эпоху, как у pref_edit_v1
ONPOLICY_REPEAT = 2                          # on-policy -- тоже 2: их мало, а сигнал главный
MAX_LEN = 8192

## 1. Карта (нужна A100 80 ГБ, память свободна)

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print(out)
name, total, used = [x.strip() for x in out.split(",")]
assert int(total) >= 79000, f"нужна карта на 80 ГБ, а здесь {total} МиБ: смените среду выполнения"
assert int(used) < 2000, f"занято {used} МиБ: остановите vLLM (!pkill -f vllm) или перезапустите среду"

## 2. Код и зависимости

In [ ]:
import os
if not os.path.exists("/content/gemma4-swe-rl"):
    !git clone -q https://github.com/aletonfdg/gemma4-swe-rl.git /content/gemma4-swe-rl
%cd /content/gemma4-swe-rl
!git pull -q
!git log --oneline | head -3
# torch и transformers в Colab уже есть; не обновляем их, чтобы не сломать совместимость
!pip install -q compressed-tensors bitsandbytes accelerate safetensors huggingface_hub kagglehub pyyaml

## 3. Drive, промпт отправки и пары
Системное сообщение во всех парах заменяется промптом отправки, чтобы обучение шло ровно под тот промпт,
с которым адаптер будет работать.

In [ ]:
import json, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/gemma_adapters"
for p in (SUBMISSION_ZIP, PAIRS, ONPOLICY):
    assert os.path.exists(f"{DRIVE}/{p}"), f"нет {DRIVE}/{p}"

SUB_DIR = "/content/submission_src"
shutil.rmtree(SUB_DIR, ignore_errors=True)
zipfile.ZipFile(f"{DRIVE}/{SUBMISSION_ZIP}").extractall(SUB_DIR)
root = next(d for d, _, files in os.walk(SUB_DIR) if "agent.yaml" in files)   # архив мог быть с лишней папкой
if root != SUB_DIR:
    tmp = SUB_DIR + "_tmp"; shutil.move(root, tmp); shutil.rmtree(SUB_DIR); shutil.move(tmp, SUB_DIR)
system = open(f"{SUB_DIR}/prompts/system.md", encoding="utf-8").read()
print(open(f"{SUB_DIR}/eval_config.yaml").read())
print("промпт:", len(system), "символов")

os.makedirs("data", exist_ok=True)
DATA = f"data/{ADAPTER}.jsonl"
EVAL_IDS = set(json.load(open("eval/mimo50.json"))["ids"])
counts = {}
with open(DATA, "w", encoding="utf-8") as out:
    for name, kinds in ((PAIRS, {"real_fix", "synthetic"}), (ONPOLICY, {"onpolicy"})):
        for line in open(f"{DRIVE}/{name}", encoding="utf-8"):
            if not line.strip():
                continue
            rec = json.loads(line)
            assert rec.get("kind") in kinds, f"{name}: неожиданный вид пары {rec.get('kind')}"
            assert rec.get("task") not in EVAL_IDS, f"задача mimo50 в парах: {rec.get('task')}"
            for side in ("chosen", "rejected"):
                assert rec[side][0]["role"] == "system", "в паре нет системного сообщения"
                rec[side][0]["content"] = system
            out.write(json.dumps(rec, ensure_ascii=False) + "\n")
            counts[rec["kind"]] = counts.get(rec["kind"], 0) + 1
print("пар по видам:", counts, "->", DATA)
assert counts.get("onpolicy", 0) >= 50 and counts.get("real_fix", 0) + counts.get("synthetic", 0) >= 150

## 4. Базовая модель в `/content/model` (если уже есть, ничего не качает)

In [ ]:
import getpass
MODEL_DIR = "/content/model"
HF_REPO = "google/gemma-4-31b-it-qat-w4a16-ct"
KAGGLE_MODEL = "google/gemma-4/transformers/gemma-4-31b-it-qat-w4a16-ct"
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    try:
        from huggingface_hub import snapshot_download
        hf_token = getpass.getpass("HF токен (Enter, если не нужен): ").strip() or None
        snapshot_download(HF_REPO, local_dir=MODEL_DIR, token=hf_token)
    except Exception as e:
        print("HF не получилось:", type(e).__name__, str(e)[:300])
        import kagglehub
        os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("KAGGLE_API_TOKEN (KGAT_...): ").strip()
        path = kagglehub.model_download(KAGGLE_MODEL)
        !cp -r "{path}"/. {MODEL_DIR}/
os.environ["GEMMA_MODEL_DIR"] = MODEL_DIR
!du -sh {MODEL_DIR}

## 5. Обучение (DPO с reference; ~4 ч; vLLM не запускать)
В логе: `пар: N ... по видам`, `reference посчитан`, потом `eval` **до и после** по видам
(`real_fix`, `synthetic`, `onpolicy`, `все`):
* `до обучения` -- у reference отступ равен 0, доля 0.00: так и должно быть;
* `после` -- `real_fix` и `synthetic` около 1.0 (у `pref_edit_v1` было 1.00);
* `onpolicy` -- eval маленький (~10 пар), шумный; хорошо, если доля >= 0.7. Если < 0.5 -- поставьте
  `ONPOLICY_REPEAT = 3` в ячейке 0 и перезапустите ячейки 0, 3, 5.

При `CUDA out of memory` -- `MAX_LEN = 6144` в ячейке 0, перезапуск 0 и 5.

In [ ]:
LOG = f"/content/{ADAPTER}.log"
!python -m train.pref_lora --data {DATA} --name {ADAPTER} --decompress --ref --epochs {EPOCHS} --max-len {MAX_LEN} --repeat real_fix={EDIT_REPEAT} synthetic={EDIT_REPEAT} onpolicy={ONPOLICY_REPEAT} 2>&1 | tee {LOG}

## 6. Сохранить адаптер, пары и лог в Drive

In [ ]:
assert os.path.exists(f"adapters/{ADAPTER}/adapter_model.safetensors"), "адаптер не сохранился, смотрите лог"
!cp -r adapters/{ADAPTER} {DRIVE}/ && cp {LOG} {DATA} {DRIVE}/
!ls -la {DRIVE}/{ADAPTER}
!grep -E "пар:|eval|reference" {LOG}

## 7. Архив отправки: основа + `adapter:` + файлы адаптера (+ время)
Результат: `gemma_adapters/upload_{ADAPTER}/submission.zip`.

In [ ]:
import re
OUT_DIR = f"/content/upload_{ADAPTER}"
shutil.rmtree(OUT_DIR, ignore_errors=True)
shutil.copytree(SUB_DIR, OUT_DIR)
shutil.rmtree(f"{OUT_DIR}/adapters", ignore_errors=True)
os.makedirs(f"{OUT_DIR}/adapters/{ADAPTER}")
for f in ("adapter_config.json", "adapter_model.safetensors"):
    shutil.copy(f"adapters/{ADAPTER}/{f}", f"{OUT_DIR}/adapters/{ADAPTER}/{f}")
agent = open(f"{OUT_DIR}/agent.yaml", encoding="utf-8").read()
agent = re.sub(r"(?m)^adapter:.*\n?", "", agent)
agent = re.sub(r"(?m)^(model:.*\n)", rf"\1adapter: {ADAPTER}\n", agent, count=1)
assert f"adapter: {ADAPTER}" in agent, "в agent.yaml нет строки model:"
open(f"{OUT_DIR}/agent.yaml", "w", encoding="utf-8").write(agent)
if MAX_TIME_MINUTES is not None:
    cfg = open(f"{OUT_DIR}/eval_config.yaml", encoding="utf-8").read()
    cfg, n = re.subn(r"(?m)^(\s*max_time_minutes:).*$", rf"\g<1> {MAX_TIME_MINUTES}", cfg)
    assert n == 1, "в eval_config.yaml нет max_time_minutes"
    open(f"{OUT_DIR}/eval_config.yaml", "w", encoding="utf-8").write(cfg)
print(agent); print(open(f"{OUT_DIR}/eval_config.yaml").read())
size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(OUT_DIR) for f in fs)
print(f"распакованный размер: {size / 2**30:.3f} ГиБ")
assert size < 3 * 2**30
ZIP = f"/content/submission_{ADAPTER}.zip"
if os.path.exists(ZIP):
    os.remove(ZIP)
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for d, _, fs in os.walk(OUT_DIR):
        for f in fs:
            p = os.path.join(d, f)
            z.write(p, os.path.relpath(p, OUT_DIR))
print(*sorted(zipfile.ZipFile(ZIP).namelist()), sep="\n")
os.makedirs(f"{DRIVE}/upload_{ADAPTER}", exist_ok=True)
shutil.copy(ZIP, f"{DRIVE}/upload_{ADAPTER}/submission.zip")
print("готово:", f"{DRIVE}/upload_{ADAPTER}/submission.zip")